# Basic example with a MODFLOW simulation

This notebook provides a minimal example of how to use `vorflow` to generate a mesh and then use it in a `MODFLOW` simulation.

In this model, we will simulate the flow between two constant-head boundaries (the two sides of the domain parallel to the y-axis), as well as a well located at (25, 25) and an impermeable fault at the center of the domain.

**Note**

This notebook uses `flopy` and the MODFLOW 6 executable (`mf6`), which are not installed with `vorflow`. Install flopy with `pip install "vorflow[examples]"`, then get `mf6` onto your PATH with `get-modflow :flopy` or `conda install -c conda-forge modflow6`.

In [ ]:
import tempfile

import matplotlib.pyplot as plt
import flopy
from shapely.geometry import LineString, Point, box

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator, set_verbosity
from vorflow.utils import build_connectivity, cells_on, faces_on

set_verbosity(0)

## Definition of the conceptual model
We will start by building the conceptual model in the `vorflow` package and add:
- The model domain
- A well (point)
- A fault (line)


In [ ]:
# Definition of geometries
domain = box(0, 0, 200, 200)
well_point = Point(25, 25)
fault_line = LineString([(100, 0), (100, 150)])

# The constant-head sides are only used to select cells after meshing,
# so they are not added to the conceptual model.
left_line = LineString([(0, 0), (0, 200)])
right_line = LineString([(200, 0), (200, 200)])

# Creation of the ConceptualMesh and addition of the geometries
cm = ConceptualMesh()
cm.add_polygon(domain, zone_id=1)
cm.add_point(
    well_point,
    point_id="Well-A",
    resolution=2,
    growth_factor=1.2,
    label="Well (5 m)"
)
cm.add_line(
    fault_line,
    line_id="Fault-1",
    resolution=1,
    is_barrier=True,
    label="Fault"
)

## Create the `MeshGenerator` object
We will create a `MeshGenerator` object and add our conceptual model to it. Once that is done, we can generate the triangular mesh with the `.generate()` method.


In [ ]:
mg = MeshGenerator(background_lc=100, conceptual_mesh=cm)
mg.generate()

## Create the `VoronoiTessellator` object
We can now create the Voronoi grid by creating a `VoronoiTessellator` object and adding our mesh and conceptual model to it.

In [ ]:
vt = VoronoiTessellator(mg, cm, clip_to_boundary=True)
gdf = vt.generate()

We can now plot the Voronoi mesh and the conceptual model with the `.plot()` method.

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
vt.plot(ax=ax, color='w', column=None, edgecolor='k');
cm.plot(ax=ax, alpha=0);
ax.legend();

## Create the `MODFLOW` model using `flopy`

We will start by setting up the MODFLOW simulation using ``flopy``.

In [ ]:
# MODFLOW 6 must be on your PATH. Uncomment to download it with flopy:
# flopy.utils.get_modflow(":flopy")

# Set up the FloPy MODFLOW 6 simulation in a temporary workspace
workspace = tempfile.TemporaryDirectory()
name = "voronoi_sim"
sim = flopy.mf6.MFSimulation(sim_name=name, exe_name="mf6", sim_ws=workspace.name)
tdis = flopy.mf6.ModflowTdis(sim, nper=1, perioddata=[(1.0, 1, 1.0)])
ims = flopy.mf6.ModflowIms(sim, complexity="MODERATE")

# Set up the Groundwater Flow (GWF) and Initial Conditions (IC) packages
gwf = flopy.mf6.ModflowGwf(sim, modelname=name)
ic = flopy.mf6.ModflowGwfic(gwf, strt=20.0) # Initial head of 20 m for all cells

# Set up basic Node Property Flow (required for flow calculations and HFB)
npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    icelltype=1,
    k=10.0,      # Default hydraulic conductivity
    save_flows=True
)

# Set up the Output Control Package (OC)
head_filename = f"{gwf.name}.hds"
budget_filename = f"{gwf.name}.cbc"

oc = flopy.mf6.ModflowGwfoc(
    gwf,
    head_filerecord=[head_filename],
    budget_filerecord=[budget_filename],
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    printrecord=[("HEAD", "ALL"), ("BUDGET", "ALL")]
)

The `DISV` package can be built either using `.to_disv()`, which returns the parameters needed for `flopy.mf6.ModflowGwfdisv()`, or using `.to_modflowgwfdisv(gwf, top=..., botm=...)`. Here, the first approach is demonstrated to visualize what is going on automatically within `.to_modflowgwfdisv(gwf, top=..., botm=...)`.

In [ ]:
# Build the DISV input from the VoronoiTessellator object
grid = vt.to_disv()

# Set up the DISV Package
nlay = 1  # Number of vertical layers
top = 100.0
botm = [0.0]  # Bottom elevation for layer 1

disv = flopy.mf6.ModflowGwfdisv(
    gwf,
    nlay=nlay,
    ncpl=grid["ncpl"],
    nvert=grid["nvert"],
    top=top,
    botm=botm,
    vertices=grid["vertices"],
    cell2d=grid["cell2d"]
)

# The code above can be replaced with
# disv = vt.to_modflowgwfdisv(gwf, top=top, botm=botm, nlay=1)

print(f"Created the DISV package with {grid['ncpl']} cells and {grid['nvert']} vertices.")

## Model boundary conditions
Now, the boundary conditions can be added to the model by first locating the boundary condition cells and then passing the conditions to the CHD and HFB packages.

In [ ]:
# MODFLOW cell ids are zero-based row positions in the grid, not index labels.
# Left Boundary Cells (20 m head)
left_cells = cells_on(gdf, left_line)

# Right Boundary Cells (10 m head)
right_cells = cells_on(gdf, right_line)

# Well Cell (5 m head)
# Find the single cell containing the well point
well_cell = cells_on(gdf, well_point)

In [ ]:
# Find all cell faces lying on the fault.
# cell_id_1/cell_id_2 are zero-based row positions, as MODFLOW expects.
connectivity = build_connectivity(gdf, center="generator")
faces_on_fault = faces_on(connectivity, fault_line, tolerance=cm.connectivity_tolerance)
assert len(faces_on_fault) > 0, "No cell faces found on the fault."

hfb_data = []
for cid1, cid2 in zip(faces_on_fault["cell_id_1"], faces_on_fault["cell_id_2"]):
    # MODFLOW 6 HFB-format: ((layer, cellid1), (layer, cellid2), factor)
    # factor = 0.0 for impermeable
    hfb_data.append(((0, cid1), (0, cid2), 0.0))

In [ ]:
# Consolidate all constant head boundary data into a single list
chd_period_data = []

# Add left boundary cells (20 m)
for cid in left_cells["cell_id"]:
    chd_period_data.append(((0, cid), 20.0))

# Add right boundary cells (10 m)
for cid in right_cells["cell_id"]:
    chd_period_data.append(((0, cid), 10.0))

# Add well cell (5 m)
for cid in well_cell["cell_id"]:
    chd_period_data.append(((0, cid), 5.0))

# Set up the CHD Package
chd = flopy.mf6.ModflowGwfchd(
    gwf, 
    stress_period_data={0: chd_period_data},
    pname="chd_main" # Explicit name prevents random index suffixes
)

# Set up the HFB Package
hfb = flopy.mf6.ModflowGwfhfb(
    gwf,
    print_input=True,
    maxhfb=len(hfb_data),
    stress_period_data=hfb_data
)

## Simulation

The model is now ready and can be run.

In [ ]:
# Write and run the simulation
sim.write_simulation()
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not run successfully:\n" + "\n".join(buff)

## Results

The heads can now be read and added to the grid GeoDataFrame for plotting.

In [ ]:
# Read the heads of the only stress period and time step
heads = gwf.output.head().get_data(kstpkper=(0, 0))

# The heads are ordered by icell2d; to_disv()["node_id"] maps them back to grid cells
layer_0_heads = dict(zip(grid["node_id"], heads[0].ravel()))
gdf["hydraulic_head"] = gdf["node_id"].map(layer_0_heads)

# Generate the plot using GeoPandas
fig, ax = plt.subplots(figsize=(10, 8))

# Plot the Voronoi polygons colored by their hydraulic head values
gdf.plot(
    column="hydraulic_head",
    cmap="viridis",
    edgecolor="black",
    linewidth=0.3,
    alpha=0.9,
    legend=True,
    legend_kwds={"label": "Hydraulic Head (m)"},
    ax=ax
)

# Formatting
ax.set_ylabel("Northing (m)")
ax.set_xlabel("Easting (m)")
ax.set_xlim(0, 200)
ax.set_ylim(0, 200)
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

# Remove the temporary model workspace
workspace.cleanup()